## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-26-15-54-13 +0000,bbc,When will Poundland shops close?,https://www.bbc.co.uk/news/articles/c6z9d3pv93...
1,2026-11-26-15-50-57 +0000,nypost,"French actor Gérard Depardieu loses appeal, wi...",https://nypost.com/2026/11/26/entertainment/ge...
2,2026-11-26-15-47-00 +0000,wsj,Marlboro Maker Altria Explores Deal for Rival ...,https://www.wsj.com/business/deals/altria-cons...
3,2026-11-26-15-46-08 +0000,bbc,Mandelson kept lobby firm shares despite confl...,https://www.bbc.co.uk/news/articles/c6808dv4e8...
4,2026-11-26-15-45-19 +0000,nypost,Ex-NFL punter flips deep red California seat b...,https://nypost.com/2026/11/26/us-news/former-v...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2512/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
44,trump,32
57,thanksgiving,27
113,new,21
352,china,14
78,he,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
359,2026-11-25-17-24-25 +0000,wapo,"Trump proposes new tariffs on China, Europe an...",https://www.washingtonpost.com/politics/2026/1...,94
385,2026-11-25-15-58-36 +0000,nyt,Trump Administration Unveils New Tariffs for C...,https://www.nytimes.com/2026/11/25/business/tr...,91
49,2026-11-26-13-58-19 +0000,nypost,Trump kicks off Thanksgiving with late night d...,https://nypost.com/2026/11/26/us-news/presiden...,81
9,2026-11-26-15-39-23 +0000,nypost,Trump tells Putin to ‘pull back all military a...,https://nypost.com/2026/11/26/us-news/trump-te...,79
180,2026-11-26-05-04-49 +0000,nypost,Russia must avoid ‘any actions’ expanding war ...,https://nypost.com/2026/11/26/us-news/russia-m...,78


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
359,94,2026-11-25-17-24-25 +0000,wapo,"Trump proposes new tariffs on China, Europe an...",https://www.washingtonpost.com/politics/2026/1...
136,65,2026-11-26-10-00-00 +0000,wsj,Democratic Sen. Chris Van Hollen wants to be S...,https://www.wsj.com/politics/policy/why-this-d...
28,64,2026-11-26-14-47-00 +0000,wsj,Macy’s Postpones Thanksgiving Day Parade as Po...,https://www.wsj.com/us-news/macys-postpones-it...
240,60,2026-11-25-22-38-54 +0000,nypost,Elon Musk’s plan for SpaceX to launch 1 millio...,https://nypost.com/2026/11/25/business/fcc-app...
120,52,2026-11-26-10-13-24 +0000,nypost,Hong Kong marks anniversary high-rise blaze th...,https://nypost.com/2026/11/26/world-news/hong-...
371,48,2026-11-25-16-43-13 +0000,nypost,"Nasdaq tumbles 4% as Nvidia leads AI sell-off,...",https://nypost.com/2026/11/25/business/nasdaq-...
180,43,2026-11-26-05-04-49 +0000,nypost,Russia must avoid ‘any actions’ expanding war ...,https://nypost.com/2026/11/26/us-news/russia-m...
35,36,2026-11-26-14-31-52 +0000,nypost,California teacher secretly filmed up sixth-gr...,https://nypost.com/2026/11/26/us-news/californ...
1,33,2026-11-26-15-50-57 +0000,nypost,"French actor Gérard Depardieu loses appeal, wi...",https://nypost.com/2026/11/26/entertainment/ge...
277,33,2026-11-25-21-31-46 +0000,nypost,Trump’s Post interview sparks effort to bring ...,https://nypost.com/2026/11/25/us-news/trumps-p...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
